## Adaptive pretraining

### Colab Setup

In [15]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Key Imports

In [16]:
import pandas as pd
import torch

from config import APT, APT_EPOCHS, IDIOMS, RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.apt_pools import build_eval_set, build_pools, build_tapt_pool
from data.loader_twd_labelled import load_splits
from models.apt import adapt
from models.frozen_probe import probe
from models.plm_finetune import finetune
from sklearn.metrics import f1_score

from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
ENC = "roberta-large"
SEEDS = SHAH_SEEDS
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

results -> /content/drive/MyDrive/thesis/results.csv | device: cuda
NVIDIA A100-SXM4-40GB


### Arms

In [ ]:
VANILLA = SHAH_PLM[ENC]["model_name"]
FOMC_POOL = build_pools(verbose=False)[0]["sentence"].to_list()
print(f"fomc pool: {len(FOMC_POOL):,} sentences")

# arm -> (sentences, epochs key, starting checkpoint, seed-dependent)
ARMS = {
    "dapt-fomc": (lambda seed: FOMC_POOL, "dapt", VANILLA, False),
    "tapt": (
        lambda seed: load_splits("benchmark", seed=seed)[0]["sentence"].to_list(),
        "tapt",
        VANILLA,
        True,
    ),
    "curated-tapt": (
        lambda seed: build_tapt_pool(seed)["sentence"].to_list(),
        "curated-tapt",
        VANILLA,
        True,
    ),
    "dapt-fomc+curated-tapt": (
        lambda seed: build_tapt_pool(seed)["sentence"].to_list(),
        "curated-tapt",
        str(RESULTS_DIR / "models" / "dapt-fomc"),
        True,
    ),
}


downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmpyp1mzga6
  meeting_minutes: 230 docs -> 47,340 sentences
  speech: 1026 docs -> 107,548 sentences
  press_conference: 63 docs -> 24,750 sentences
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
fomc pool: 164,688 sentences


### Continued pretraining

In [ ]:
for arm, (pool_fn, epochs_key, start, per_seed) in ARMS.items():
    for seed in SEEDS if per_seed else [None]:
        name = f"{arm}-s{seed}" if per_seed else arm
        save_dir = str(RESULTS_DIR / "models" / name)
        if os.path.isdir(save_dir):
            print(f"{name}: already adapted, skipping")
            continue
        sentences = pool_fn(seed)
        print(f"{name}: {len(sentences):,} sentences", flush=True)
        adapt(
            sentences,
            model_name=start,
            epochs=APT_EPOCHS[epochs_key],
            save_dir=save_dir,
            device=DEVICE,
            verbose=True,
            **APT,
        )

dapt-fomc: already adapted, skipping
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
tapt-s5768: 1,984 sentences


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

    tokenizing 1,984 sentences...
    training: 62 batches/epoch x 100 epoch(s), 6,200 updates at effective batch 32
    epoch 0: mlm loss 1.3848
    epoch 1: mlm loss 1.3076
    epoch 2: mlm loss 1.2444
    epoch 3: mlm loss 1.2045
    epoch 4: mlm loss 1.2175
    epoch 5: mlm loss 1.1943
    epoch 6: mlm loss 1.1845
    epoch 7: mlm loss 1.1370
    epoch 8: mlm loss 1.1490
    epoch 9: mlm loss 1.1858
    epoch 10: mlm loss 1.1359
    epoch 11: mlm loss 1.0076
    epoch 12: mlm loss 0.9865
    epoch 13: mlm loss 0.9963
    epoch 14: mlm loss 0.9660
    epoch 15: mlm loss 0.9704
    epoch 16: mlm loss 0.9001
    epoch 17: mlm loss 0.9067
    epoch 18: mlm loss 0.8982
    epoch 19: mlm loss 0.8696
    epoch 20: mlm loss 0.8466
    epoch 21: mlm loss 0.8917
    epoch 22: mlm loss 0.8253
    epoch 23: mlm loss 0.8358
    epoch 24: mlm loss 0.7908
    epoch 25: mlm loss 0.7827
    epoch 26: mlm loss 0.7598
    epoch 27: mlm loss 0.7680
    epoch 28: mlm loss 0.7445
    epoch 29: mlm loss 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    saved -> /content/drive/MyDrive/thesis/models/tapt-s5768
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
tapt-s78516: 1,984 sentences


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

    tokenizing 1,984 sentences...
    training: 62 batches/epoch x 100 epoch(s), 6,200 updates at effective batch 32
    epoch 0: mlm loss 1.3969
    epoch 1: mlm loss 1.2947
    epoch 2: mlm loss 1.2618
    epoch 3: mlm loss 1.2514
    epoch 4: mlm loss 1.1817
    epoch 5: mlm loss 1.2080
    epoch 6: mlm loss 1.1685
    epoch 7: mlm loss 1.1376
    epoch 8: mlm loss 1.1102
    epoch 9: mlm loss 1.0853
    epoch 10: mlm loss 1.0698
    epoch 11: mlm loss 1.0279
    epoch 12: mlm loss 1.0137
    epoch 13: mlm loss 0.9746
    epoch 14: mlm loss 1.0094
    epoch 15: mlm loss 0.9621
    epoch 16: mlm loss 0.9368
    epoch 17: mlm loss 0.9016
    epoch 18: mlm loss 0.9143
    epoch 19: mlm loss 0.8897
    epoch 20: mlm loss 0.8447
    epoch 21: mlm loss 0.8660
    epoch 22: mlm loss 0.8445
    epoch 23: mlm loss 0.8639
    epoch 24: mlm loss 0.7889
    epoch 25: mlm loss 0.8046
    epoch 26: mlm loss 0.7883
    epoch 27: mlm loss 0.7629
    epoch 28: mlm loss 0.8064
    epoch 29: mlm loss 

### Held-out MLM loss

In [17]:
# a fixed slice of the filtered corpus, subtracted from every pool and holding no
# labelled sentence, so no arm has trained on it and it needs no seed.
eval_on = build_eval_set()["sentence"].to_list()

from torch.utils.data import DataLoader
from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
)



def mlm_loss(model_path, seed=0):
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForMaskedLM.from_pretrained(model_path).to(DEVICE).eval()
    enc = tok(eval_on, truncation=True, max_length=APT["max_len"])
    rows = [
        {"input_ids": i, "attention_mask": m}
        for i, m in zip(enc["input_ids"], enc["attention_mask"])
    ]
    dl = DataLoader(
        rows,
        batch_size=APT["batch_size"],
        collate_fn=DataCollatorForLanguageModeling(
            tok, mlm_probability=APT["mlm_probability"]
        ),
    )
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            total += model(**batch).loss.item()
            n += 1
    del model
    torch.cuda.empty_cache()
    return total / n


paths = {ENC: VANILLA}
for arm, (_, _, _, per_seed) in ARMS.items():
    for seed in SEEDS if per_seed else [None]:
        name = f"{arm}-s{seed}" if per_seed else arm
        path = str(RESULTS_DIR / "models" / name)
        if os.path.isdir(path):
            paths[name] = path

rows = [dict(model=k, mlm_loss=round(mlm_loss(v), 4)) for k, v in paths.items()]
for r in rows:
    print(f"{r['model']}: {r['mlm_loss']}")

pd.DataFrame(rows).to_csv(RESULTS_DIR / "mlm_loss.csv", index=False)
print("saved ->", RESULTS_DIR / "mlm_loss.csv")


Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
eval set: 1,248 sentences, train under every seed


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

roberta-large: 1.3456
dapt-fomc: 1.0607
saved -> /content/drive/MyDrive/thesis/mlm_loss.csv


### Masked-idiom probe

In [ ]:
from transformers import pipeline


def probe_idioms(model_path):
    mlm = pipeline("fill-mask", model=model_path, device=0 if DEVICE == "cuda" else -1)
    mask = mlm.tokenizer.mask_token
    rows = []
    for phrase, gold in IDIOMS:
        top5 = [
            r["token_str"].strip().lower()
            for r in mlm(phrase.replace("[MASK]", mask), top_k=5)
        ]
        rows.append(
            dict(phrase=phrase, gold=gold, hit=gold.lower() in top5, top5="|".join(top5))
        )
    del mlm
    torch.cuda.empty_cache()
    return rows


models = {ENC: VANILLA}
for arm, (_, _, _, per_seed) in ARMS.items():
    name = f"{arm}-s{SEEDS[0]}" if per_seed else arm
    models[arm] = str(RESULTS_DIR / "models" / name)

records = []
for label, path in models.items():
    rows = probe_idioms(path)
    records += [dict(model=label, **r) for r in rows]
    print(f"{label}: {sum(r['hit'] for r in rows)}/{len(rows)}", flush=True)

idf = pd.DataFrame(records)
idf.to_csv(RESULTS_DIR / "idioms.csv", index=False)
print("saved ->", RESULTS_DIR / "idioms.csv")

### Fine-tune

In [ ]:
cfg = SHAH_PLM[ENC]

for arm, (_, _, _, per_seed) in ARMS.items():
    for seed in SEEDS:
        model_key = f"{arm}:{ENC}"
        if already_done(OUT, force=FORCE, model=model_key, corpus="twd", seed=seed):
            print(f"{model_key} seed {seed}: already done, skipping")
            continue
        name = f"{arm}-s{seed}" if per_seed else arm
        train, test = load_splits("benchmark", seed=seed)
        model, tok_, metrics = finetune(
            train,
            model_name=str(RESULTS_DIR / "models" / name),
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=seed,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=model_key,
            corpus="twd",
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{model_key} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()

### Frozen probe

In [ ]:
for arm, (_, _, _, per_seed) in ARMS.items():
    for seed in SEEDS:
        model_key = f"frozen-{arm}:{ENC}"
        if already_done(OUT, force=FORCE, model=model_key, corpus="twd", seed=seed):
            print(f"{model_key} seed {seed}: already done, skipping")
            continue
        name = f"{arm}-s{seed}" if per_seed else arm
        train, test = load_splits("benchmark", seed=seed)
        pred = probe(
            train,
            test,
            model_name=str(RESULTS_DIR / "models" / name),
            device=DEVICE,
            seed=seed,
        )
        true = test["label"].to_list()
        save_result(
            OUT,
            model=model_key,
            corpus="twd",
            seed=seed,
            epochs="",
            weighted_f1=round(f1_score(true, pred, average="weighted"), 4),
            macro_f1=round(f1_score(true, pred, average="macro"), 4),
        )
        print(f"{model_key} seed {seed}: macro={f1_score(true, pred, average='macro'):.4f}")
